In [ ]:
################# Adamts4 project, to be submitted to NatComm ####################
###################### 1_Classification/Annotation & Velocity ####################

In [ ]:
import os
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import cellrank as cr
import scvelo as scv
import scanpy.external as sce
import harmonypy as hm
import bbknn
import scvi
import gseapy as gp
from scipy import sparse
from scipy.stats import zscore
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.sparse import issparse
from sklearn.preprocessing import LabelEncoder
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.pyplot import rc_context
from pathlib import Path

warnings.filterwarnings("ignore")

sc.settings.verbosity = 3
sc.settings.n_jobs = os.cpu_count()
sc.set_figure_params(figsize=(10, 10), frameon=False)
scv.set_figure_params()
plt.rcParams['axes.grid'] = False

plot_dir = '/Users/aly/Downloads/2024_Ali_scRNAseq/2024_Mahsa_scRNAseq/Analysis/0_260318_Final'
os.makedirs(plot_dir, exist_ok=True)

In [ ]:
import multiprocessing
print("Available CPU cores:", multiprocessing.cpu_count())

In [ ]:
adata = sc.read_h5ad('/Users/aly/Downloads/2024_Ali_scRNAseq/2024_Mahsa_scRNAseq/ScanpyObjects/241129_Bleo_Harmony_v2.h5ad')
adata

In [ ]:
import matplotlib.colors as mcolors
keys = ['leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'leiden_1.0', 'leiden_1.4']
final_hex = list(mcolors.TABLEAU_COLORS.values()) + list(mcolors.XKCD_COLORS.values())

for k in keys:
    adata.obs[k] = pd.Categorical(adata.obs[k], categories=sorted(adata.obs[k].unique(), key=int))
    adata.uns[f'{k}_colors'] = final_hex[:len(adata.obs[k].cat.categories)]

sc.pl.umap(adata, color=keys, size=15, ncols=3, legend_loc='on data', frameon=False)

In [ ]:
annotation_dict = {
    "AF1": ["Inmt", "Npnt", "Limch1", "G0s2", "Nebl", "Tcf21", "Fgf10", "Col13a1", "Scn3a", "Pcolce2", "Gyg", "Has1"],
    "AF2": ["Serpinf1", "Scara5", "Adh7", "Pi16", "Dcn", "Ly6a", "Wif1", "Col14a1", "Sparc", "Lum"],
    "PeriFB": ["Mustn1", "Aspn", "Hhip", "Fgf18", "Agt", "Enpp2", "P2ry14", "Pdfrb"],
    "SMCs": ["Tagln", "Acta2", "Cnn1", "Notch3", "Fst", "Myh11", "Gucy1a1", "Myl6", "Hspb1", "Mygz"],
    "PeriC": ["Crip1", "Cspg4", "Kcnk3", "Gucy1a3", "Gucy1b3", "Emid1", "Abcc9", "Rgs5"],
    "Meso": ["Slurp1", "Msln", "Gpc3", "Gsta3", "Upk3b", "Wt1"],
    "MyoFB": ["Lcn2", "Spp1", "Cthrc1", "Grem1", "Piezo2", "Cck", "Nrep", "Bpgm", "Col7a1", "Postn", "Fn1"],
    "ProlifFB": ["Mki67", "Col1a1", "Top2a", "Pdgfra", "Cenpf", "Stmn1"],
    "AT2": ["Sftpb", "Sftpc", "Napsa", "Lyz1", "Lyz2", "Scd1", "Chil1", "Etv5", "Abca3"],
    "AT1": ["Rtkn2", "Spock2", "Hopx", "Pdpn", "Ager", "Akap5", "Sema3a", "Scnn1g", "Vegfa", "Cav1"],
    "ADI": ["Krt8", "Areg", "Sfn", "Clu", "Ndnf", "Krt18", "Ano1", "S100a14", "Krt19"],
    "Club": ["Scgb3a1", "Reg3g", "Cyp2f2", "Scgb3a2", "Scgb1a1", "Krt15"],
    "Ciliated": ["Tppp3", "Cdkn1c", "Dynlrb2", "Tmem212", "Foxj1", "Ccdc153"],
    "Endo": ["Pecam1", "Vwf", "Esam", "Cdh5", "Kdr", "Flt1"],
    "gCap": ["Kit", "Gpihbp1", "Itga6", "Peg3", "Plvap"],
    "aCap": ["Emp2", "Car4", "Igfbp7", "Ednrb", "Hbb-bs", "Aplnr"],
    "AlvMc": ["Chil3", "Plet1", "Lpl", "Ctsd", "Atp6v0d2", "Marco", "Siglecf"],
    "Neu": ["Gm5416", "Asprv1", "Ngp", "Ltf", "Stfa3", "Camp", "S100a8", "S100a9"],
    "IntMc": ["Apoe", "C1qc", "C1qb", "C1qa", "Pf4", "Mrc1", "Cd163"],
    "MastBa2": ["Ccl3", "Ccl4", "Il6", "Ifitm1", "Mcpt8", "Fcer1a", "Kit"],
    "DC": ["Traf1", "Ccr7", "Ccl5", "Fscn1", "Epsti1", "Cd74", "H2-Ab1"],
    "TC": ["Mier1", "Ccr9", "Dntt", "Rag1", "Tifa", "Tcf7", "Lef1", "Ly6c2", "Dusp10", "Ms4a6b", "Cd8a", "Cd3e"],
    "BC": ["Ly6d", "Cd74", "Cd79a", "Ms4a1", "Igkc", "Tifa", "Cd79b"],
    "TranFBs": ["Col28a1", "Sfrp1", "Runx1", "Hp", "Inhba"]
}

def annotate_clusters(adata, cluster_key, annotation_dict, top_n=300, min_matches=3):
    sc.tl.rank_genes_groups(adata, cluster_key, method="t-test")
    res = adata.uns["rank_genes_groups"]
    mapping = {}

    for cluster in res["names"].dtype.names:
        top_genes = res["names"][cluster][:top_n]
        scores = {ct: len(set(top_genes) & set(mk)) for ct, mk in annotation_dict.items()}
        best_ct = max(scores, key=scores.get)
        mapping[cluster] = best_ct if scores[best_ct] >= min_matches else "Unknown"

    adata.obs["celltype"] = adata.obs[cluster_key].map(mapping)
    return adata

adata = annotate_clusters(adata, "leiden_0.6", annotation_dict)
sc.pl.umap(adata, color="celltype", legend_loc="on data", size=10, frameon=False)

In [ ]:
sc.pl.umap(adata, color=['sample', 'celltype'], size=15, alpha=0.5)

In [ ]:
samples = adata.obs['sample'].cat.categories
n_cols = 2
n_rows = int(np.ceil(len(samples) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows), facecolor='white')

for i, s in enumerate(samples):
    ax = axes.flatten()[i]
    sc.pl.umap(adata[adata.obs['sample'] == s], color='celltype', title=f"Sample: {s}", 
               ax=ax, show=False, size=40, frameon=False)
    ax.set(xlabel="", ylabel="")

for j in range(i + 1, len(axes.flatten())): axes.flatten()[j].axis('off')

plt.savefig(f"{plot_dir}/1.umap_celltype_split_sample.png", bbox_inches='tight', facecolor='white', dpi=300)
plt.show()

In [ ]:
sc.pl.umap(adata, color=['Fgf10', 'Serpinf1','Cthrc1', 'Fgf18'], cmap='twilight', ncols=2, size= 25, wspace=0.1)

In [ ]:
sc.pl.dotplot(adata, var_names=['Scube2', 'Serpinf1', 'Cthrc1', 'Mustn1'], groupby='celltype', cmap='RdYlBu_r')

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib
import scanpy as sc
import scvelo as scv
from matplotlib import gridspec, cm

matplotlib.use('Agg')

for sample in adata.obs['sample'].unique():
    subset = adata[adata.obs['sample'] == sample].copy()
    sample_dir = os.path.join(plot_dir, sample)
    os.makedirs(sample_dir, exist_ok=True)

    sc.pp.neighbors(subset, n_neighbors=30, n_pcs=20)
    scv.pp.moments(subset)
    scv.tl.velocity(subset, n_jobs=10)
    scv.tl.velocity_graph(subset, n_jobs=10)
    scv.tl.velocity_pseudotime(subset)
    scv.tl.paga(subset, groups='celltype')

    # Velocity Grid
    scv.pl.velocity_embedding_grid(subset, basis='umap', color='celltype', alpha=0.5, arrow_size=1, arrow_length=4, title=f'Velocity of {sample}', size=45, show=False)
    plt.savefig(os.path.join(sample_dir, 'velocity_embedding_grid.png'), bbox_inches='tight', dpi=1200, facecolor='white')
    plt.close()

    # Velocity Stream
    scv.pl.velocity_embedding_stream(subset, basis='umap', color='celltype', alpha=0.7, arrow_size=0.8, size=95, title=f'Velocity Stream {sample}', show=False)
    plt.savefig(os.path.join(sample_dir, 'velocity_embedding_stream.png'), bbox_inches='tight', dpi=1200, facecolor='white')
    plt.close()

    # Pseudotime Scatter
    scv.pl.scatter(subset, color='velocity_pseudotime', cmap='gnuplot', size=45, show=False)
    plt.savefig(os.path.join(sample_dir, 'velocity_pseudotime.png'), bbox_inches='tight', dpi=800, facecolor='white')
    plt.close()

    # PAGA
    scv.pl.paga(subset, basis='umap', size=45, alpha=0.4, min_edge_width=1, node_size_scale=1, transition_probabilities=True, arrow_size=1.5, show=False)
    plt.savefig(os.path.join(sample_dir, 'velocity_PAGA.png'), bbox_inches='tight', dpi=800, facecolor='white')
    plt.close()

    # Pseudotime Stream
    fig = plt.figure(figsize=(7, 4), facecolor='white')
    gs = gridspec.GridSpec(1, 2, width_ratios=[0.9, 0.05], wspace=0.2)
    ax = plt.subplot(gs[0])
    scv.pl.velocity_embedding_stream(subset, basis='umap', ax=ax, show=False, legend_loc='none', colorbar=None, size=195)
    
    pseudotime = subset.obs['velocity_pseudotime']
    norm = plt.Normalize(vmin=pseudotime.min(), vmax=pseudotime.max())
    cmap = cm.get_cmap('viridis')
    ax.scatter(subset.obsm['X_umap'][:, 0], subset.obsm['X_umap'][:, 1], c=pseudotime, cmap=cmap, norm=norm, s=25)
    ax.set(xticks=[], yticks=[], title=f'Velocity Pseudotime {sample}')
    
    fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), cax=plt.subplot(gs[1]))
    plt.savefig(os.path.join(sample_dir, 'velocity_pseudotimestream.png'), bbox_inches='tight', dpi=800, facecolor='white')
    plt.close()

In [ ]:
import os, numpy as np, scipy.ndimage, matplotlib.pyplot as plt, scvelo as scv, matplotlib.colors as mcolors
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

fixed_gene_count, fig_height = 75, 20

for sample in adata.obs['sample'].unique():
    subset = adata[adata.obs['sample'] == sample].copy()
    scv.pp.neighbors(subset, n_neighbors=30, n_pcs=20)
    scv.pp.moments(subset, n_neighbors=30, n_pcs=20)
    scv.tl.recover_dynamics(subset, n_jobs=12)

    df = subset.var[subset.var['fit_likelihood'] > 0.1] if 'fit_likelihood' in subset.var else subset.var
    filtered_genes = df['fit_likelihood'].sort_values(ascending=False).index[:fixed_gene_count].tolist()
    
    if len(filtered_genes) < fixed_gene_count:
        extra = [g for g in subset.var_names if g not in filtered_genes][:fixed_gene_count - len(filtered_genes)]
        filtered_genes += extra

    scv.tl.velocity(subset, var_names=filtered_genes, mode='dynamical', n_jobs=12)
    scv.tl.velocity_graph(subset, n_jobs=12)
    scv.tl.latent_time(subset)

    cells_order = subset.obs.sort_values('latent_time').index
    latent_time, clusters = subset.obs.loc[cells_order, 'latent_time'].values, subset.obs.loc[cells_order, 'celltype']
    cluster_lut = dict(zip(clusters.cat.categories, subset.uns['celltype_colors']))

    hm = scv.pl.heatmap(subset, var_names=filtered_genes, sortby='latent_time', col_color=None, 
                        n_convolve=max(10, int(subset.n_obs/5)), color_map='inferno', figsize=(14, fig_height), show=False)
    fig, ax_hm = hm.fig, hm.ax_heatmap
    pos = ax_hm.get_position()

    ax_stripe = fig.add_axes([pos.x0, pos.y1 + 0.005, pos.width, 0.02])
    ax_stripe.imshow(np.array([mcolors.to_rgb(cluster_lut[cl]) for cl in clusters])[None, :, :], aspect='auto')
    ax_stripe.axis('off')

    ax_curve = fig.add_axes([pos.x0, pos.y1 + 0.03, pos.width, 0.08])
    x_norm = np.linspace(0, 1, subset.n_obs)
    for cl in clusters.unique():
        dens, _ = np.histogram(latent_time[clusters == cl], bins=75, density=True)
        y = np.interp(np.linspace(0, 75, subset.n_obs), np.arange(75), scipy.ndimage.gaussian_filter1d(dens, 2))
        y = (y / y.max() * 0.8) if y.max() > 0 else y
        ax_curve.plot(x_norm, y, color=cluster_lut[cl], lw=2)
        ax_curve.fill_between(x_norm, 0, y, color=cluster_lut[cl], alpha=0.1)

    ax_curve.axis('off')
    ax_curve.legend(handles=[Patch(color=c, label=l) for l, c in cluster_lut.items()], frameon=False, fontsize=6, loc='upper right')
    plt.savefig(os.path.join(plot_dir, f'Latent_{sample}_velocity_v2.png'), dpi=500, bbox_inches='tight')
    plt.close()

In [ ]:
import os, numpy as np, scipy.ndimage, matplotlib.pyplot as plt, scvelo as scv, matplotlib.colors as mcolors
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

fixed_gene_count, fig_height = 100, 25
scv.pp.neighbors(adata, n_neighbors=30, n_pcs=20)
scv.pp.moments(adata, n_neighbors=30, n_pcs=20)
scv.tl.recover_dynamics(adata, n_jobs=10)

df = adata.var[adata.var['fit_likelihood'] > 0.1] if 'fit_likelihood' in adata.var else adata.var
filtered_genes = df['fit_likelihood'].sort_values(ascending=False).index[:fixed_gene_count].to_list()

if len(filtered_genes) < fixed_gene_count:
    extra = [g for g in adata.var_names if g not in filtered_genes][:fixed_gene_count - len(filtered_genes)]
    filtered_genes += extra

scv.tl.velocity(adata, var_names=filtered_genes, mode='dynamical', n_jobs=10)
scv.tl.velocity_graph(adata, n_jobs=10)
scv.tl.latent_time(adata)

cells_order = adata.obs.sort_values('latent_time').index
latent_time, clusters = adata.obs.loc[cells_order, 'latent_time'].values, adata.obs.loc[cells_order, 'celltype']
cluster_lut = dict(zip(clusters.cat.categories, adata.uns['celltype_colors']))

hm = scv.pl.heatmap(adata, var_names=filtered_genes, sortby='latent_time', col_color=None, 
                    n_convolve=max(20, int(adata.n_obs/5)), color_map='inferno', figsize=(14, fig_height), show=False)
fig, ax_hm = hm.fig, hm.ax_heatmap

ax_stripe = fig.add_axes([ax_hm.get_position().x0, ax_hm.get_position().y1 + 0.005, ax_hm.get_position().width, 0.02])
ax_stripe.imshow(np.array([mcolors.to_rgb(cluster_lut[cl]) for cl in clusters])[None, :, :], aspect='auto')
ax_stripe.axis('off')

ax_curve = fig.add_axes([ax_hm.get_position().x0, ax_hm.get_position().y1 + 0.03, ax_hm.get_position().width, 0.08])
x_norm = np.linspace(0, 1, adata.n_obs)
for cl in clusters.unique():
    dens, _ = np.histogram(latent_time[clusters == cl], bins=100, density=True)
    y = np.interp(np.linspace(0, 100, adata.n_obs), np.arange(100), scipy.ndimage.gaussian_filter1d(dens, 2))
    y = (y / y.max() * 0.8) if y.max() > 0 else y
    ax_curve.plot(x_norm, y, color=cluster_lut[cl], lw=2)
    ax_curve.fill_between(x_norm, 0, y, color=cluster_lut[cl], alpha=0.1)

ax_curve.axis('off')
ax_curve.legend(handles=[Patch(color=c, label=l) for l, c in cluster_lut.items()], frameon=False, loc='upper right')
plt.savefig(os.path.join(plot_dir, 'Latent_integrated_velocity_v2.png'), dpi=500, bbox_inches='tight')
plt.close()

In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib as mpl, os
cmap, lv = mpl.cm.inferno, np.linspace(0,1,500)
fig, ax = plt.subplots(figsize=(6,1), facecolor='white')
for i, v in enumerate(lv):
    h = 1 - i/len(lv)
    ax.fill_between([i/len(lv), (i+1)/len(lv)], -h/2, h/2, color=cmap(v), edgecolor='none')
ax.set(xlim=(0,1), ylim=(-0.6,0.6), xticks=[], yticks=[])
for x in np.linspace(0,1,5):
    val = np.round(latent_time.min() + x * (latent_time.max() - latent_time.min()), 2)
    ax.text(x, -0.65, f"{val}", ha='center', va='center', fontsize=11)
ax.text(0.5, 0.7, 'Latent time', ha='center', va='center', fontsize=12, fontweight='bold')
for sp in ax.spines.values(): sp.set_visible(False)

os.makedirs(plot_dir, exist_ok=True)
fig.savefig(os.path.join(plot_dir, 'Latent_time_triangle_horizontal.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()